# 05 — Control plane: NVIDIA playbook + eugr (not random Compose)

**Milestone (L4):** you understand **why** ad-hoc `docker compose up` on two nodes failed—and what replaces it.

**Reality check:** many of us (Gemini-assisted or not) tried **Compose + Ray** without the **NVIDIA** physical/network contract or **eugr**’s `launch-cluster.sh` integration. Dependencies explode across:

- **Host** drivers / RDMA / GID
- **Network** namespace (`--network host` is common for a reason)
- **Ray** ports and head/worker addressing
- **NCCL** env for RoCE
- **vLLM** TP and weight paths **inside** mounts

**Docker does not replace the stack** — it packages a **slice** of userspace. You still need L1–L3 correct and NCCL sane.

**Upstream:**
- [NVIDIA connect-two-sparks playbook](https://github.com/NVIDIA/dgx-spark-playbooks/tree/main/nvidia/connect-two-sparks)
- [eugr/spark-vllm-docker](https://github.com/eugr/spark-vllm-docker) (`build-and-copy.sh`, `launch-cluster.sh`)

## Checklist

- [ ] Passwordless **SSH from head → worker on cluster IPs** (`10.0.0.x`), not only from laptop → LAN.
- [ ] Clone **`spark-vllm-docker`** on the **head** (and understand `build-and-copy.sh -c` for image sync).
- [ ] Read eugr **`docs/NETWORKING.md`** if linked from their README.
- [ ] Decide: **Ray** (default recipes) vs **`--no-ray`** for isolation debugging.
- [ ] Always run **`launch-cluster.sh` on the head Spark**, not your laptop.

**Green signal:** you can `ssh user@10.0.0.2` from `10.0.0.1` without password; Docker image exists on both nodes if using cluster build copy.

In [ ]:
# ON HEAD — passwordless SSH to worker on *cluster* IP?
import shutil
import subprocess

WORKER_SSH = "youruser@10.0.0.2"  # <-- cluster IP, not LAN

ssh = shutil.which("ssh")
if ssh:
    r = subprocess.run(
        [ssh, "-o", "BatchMode=yes", "-o", "ConnectTimeout=5", WORKER_SSH, "hostname"],
        capture_output=True,
        text=True,
    )
    print(("✅" if r.returncode == 0 else "❌"), (r.stdout or r.stderr).strip())
else:
    print("ssh not found")

## Next

[`06_nccl_roce_gpu_collectives.ipynb`](06_nccl_roce_gpu_collectives.ipynb) — prove **NCCL** can use **RoCE** before blaming vLLM.